<a href="https://colab.research.google.com/github/Prajnaa002/Applied_Linear_Algebra/blob/main/ALA_Lab_Assignment2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#ALGORITHM 1 : Token embedding

In [59]:
import numpy as np

def token_embedding(W_e, v):
    """
    Convert a token ID into its embedding vector.

    Parameters:
        W_e: Embedding matrix (d_e x N_V)
        v: Token ID

    Returns:
        Embedding vector of dimension d_e
    """

    # Validate token ID
    if not isinstance(v, (int, np.integer)):
        raise TypeError("Token ID must be an integer")

    if v < 0 or v >= W_e.shape[1]:
        raise ValueError("Invalid token ID")

    # Retrieve embedding vector
    e = W_e[:, v]

    return e


# Example
W_e = np.array([
    [0.1, 0.2, 0.3, 0.4, 0.5],
    [0.6, 0.7, 0.8, 0.9, 1.0],
    [1.1, 1.2, 1.3, 1.4, 1.5]
])

v = 2

embedding = token_embedding(W_e, v)

print("Token ID:", v)
print("Embedding Vector:", embedding)

Token ID: 2
Embedding Vector: [0.3 0.8 1.3]


In [60]:
import unittest
import numpy as np

class TestTokenEmbedding(unittest.TestCase):

    def setUp(self):
        self.W_e = np.array([
            [0.1, 0.2, 0.3, 0.4, 0.5],
            [0.6, 0.7, 0.8, 0.9, 1.0],
            [1.1, 1.2, 1.3, 1.4, 1.5]
        ])

    def test_valid_token(self):
        result = token_embedding(self.W_e, 2)
        expected = np.array([0.3, 0.8, 1.3])
        np.testing.assert_array_equal(result, expected)

    def test_first_token(self):
        result = token_embedding(self.W_e, 0)
        expected = np.array([0.1, 0.6, 1.1])
        np.testing.assert_array_equal(result, expected)

    def test_last_token(self):
        result = token_embedding(self.W_e, 4)
        expected = np.array([0.5, 1.0, 1.5])
        np.testing.assert_array_equal(result, expected)

    def test_invalid_token(self):
        with self.assertRaises(ValueError):
            token_embedding(self.W_e, 5)

    def test_negative_token(self):
        with self.assertRaises(ValueError):
            token_embedding(self.W_e, -1)

    def test_non_integer_token(self):
        with self.assertRaises(TypeError):
            token_embedding(self.W_e, 2.5)


if __name__ == "__main__":
    unittest.main(argv=['first-arg-is-ignored'], exit=False)

..................
----------------------------------------------------------------------
Ran 18 tests in 0.021s

OK


#ALGORITHM 6 : layer_norm

In [61]:
import numpy as np

def layer_norm(e, gamma, beta, epsilon=1e-5):
    """
    Performs Layer Normalization.

    Parameters:
        e      : Input activation vector
        gamma  : Learnable scale parameter
        beta   : Learnable offset parameter
        epsilon: Small constant for numerical stability

    Returns:
        Normalized activation vector
    """

    e = np.asarray(e)
    gamma = np.asarray(gamma)
    beta = np.asarray(beta)

    if e.ndim != 1:
        raise ValueError("Input must be a 1D vector")

    if e.shape != gamma.shape or e.shape != beta.shape:
        raise ValueError("Input, gamma and beta must have same shape")

    mean = np.mean(e)

    variance = np.mean((e - mean) ** 2)

    normalized = (e - mean) / np.sqrt(variance + epsilon)

    output = normalized * gamma + beta

    return output

e = np.array([2.0, 4.0, 6.0, 8.0])

gamma = np.ones(4)
beta = np.zeros(4)

result = layer_norm(e, gamma, beta)

print("Input:", e)
print("Normalized Output:", result)
print("Output Mean:", np.mean(result))
print("Output Variance:", np.var(result))

Input: [2. 4. 6. 8.]
Normalized Output: [-1.34163944 -0.44721315  0.44721315  1.34163944]
Output Mean: 0.0
Output Variance: 0.999998000004


In [62]:
import unittest
import numpy as np


class TestLayerNorm(unittest.TestCase):

    def setUp(self):
        self.e = np.array([2.0, 4.0, 6.0, 8.0])
        self.gamma = np.ones(4)
        self.beta = np.zeros(4)

    # Test 1: Check normalized output
    def test_normalization(self):
        result = layer_norm(
            self.e, self.gamma, self.beta
        )

        expected = np.array([
            -1.34164, -0.44721, 0.44721, 1.34164
        ])

        np.testing.assert_allclose(
            result, expected, atol=1e-4
        )

    # Test 2: Check output mean
    def test_zero_mean(self):
        result = layer_norm(
            self.e, self.gamma, self.beta
        )

        self.assertAlmostEqual(np.mean(result), 0.0, places=5)

    # Test 3: Check output variance
    def test_unit_variance(self):
        result = layer_norm(
            self.e, self.gamma, self.beta
        )

        self.assertAlmostEqual(np.var(result), 1.0, places=4)

    # Test 4: Check custom gamma and beta
    def test_scale_and_shift(self):
        gamma = np.array([2.0, 2.0, 2.0, 2.0])
        beta = np.array([1.0, 1.0, 1.0, 1.0])

        result = layer_norm(self.e, gamma, beta)

        expected = (
            layer_norm(self.e, self.gamma, self.beta) * 2 + 1
        )

        np.testing.assert_allclose(result, expected)

    # Test 5: Check mismatched dimensions
    def test_invalid_dimensions(self):
        gamma = np.ones(3)
        beta = np.zeros(4)

        with self.assertRaises(ValueError):
            layer_norm(self.e, gamma, beta)

    # Test 6: Check multidimensional input
    def test_multidimensional_input(self):
        e = np.array([[1, 2], [3, 4]])

        with self.assertRaises(ValueError):
            layer_norm(e, np.ones((2, 2)), np.zeros((2, 2)))


if __name__ == "__main__":
    unittest.main(argv=['first-arg-is-ignored'], exit=False)

..................
----------------------------------------------------------------------
Ran 18 tests in 0.022s

OK


#ALGORITHM 4 : ATTENTION

In [52]:
import numpy as np

def softmax(X, axis=0):
    X = X - np.max(X, axis=axis, keepdims=True)
    exp_X = np.exp(X)
    return exp_X / np.sum(exp_X, axis=axis, keepdims=True)


def attention(X, Z, w: tuple):

    Wq, bq, Wk, bk, Wv, bv = w

    # Step 1: Query
    Q = Wq @ X + bq

    # Step 2: Key
    K = Wk @ Z + bk

    # Step 3: Value
    V = Wv @ Z + bv

    # Step 4: Attention Scores
    S = K.T @ Q

    # Step 5: Apply Causal Mask
    Mask = np.tril(np.ones(S.shape, dtype=bool))
    S = np.where(Mask, S, -np.inf)

    # Step 6: Scaled Softmax
    A = softmax(S / np.sqrt(Q.shape[0]), axis=0)

    # Step 7: Calculate Output
    return V @ A

In [53]:
d_x, d_z = 2, 2
l_x, l_z = 3, 3
d_attn = 3
d_out = 2


def test_attention():
    np.random.seed(42)

    X = np.random.rand(d_x, l_x)
    Z = np.random.rand(d_z, l_z)

    Wq = np.random.rand(d_attn, d_x)
    bq = np.random.rand(d_attn, 1)

    Wk = np.random.rand(d_attn, d_z)
    bk = np.random.rand(d_attn, 1)

    Wv = np.random.rand(d_out, d_z)
    bv = np.random.rand(d_out, 1)

    w = (Wq, bq, Wk, bk, Wv, bv)

    c = attention(X, Z, w)

    return c


def test_zeros():
    X = np.zeros((d_x, l_x))
    Z = np.zeros((d_z, l_z))

    Wq = np.zeros((d_attn, d_x))
    bq = np.zeros((d_attn, 1))

    Wk = np.zeros((d_attn, d_z))
    bk = np.zeros((d_attn, 1))

    Wv = np.zeros((d_out, d_z))
    bv = np.zeros((d_out, 1))

    w = (Wq, bq, Wk, bk, Wv, bv)

    c = attention(X, Z, w)

    return c


def test_ones():
    X = np.ones((d_x, l_x))
    Z = np.ones((d_z, l_z))

    Wq = np.ones((d_attn, d_x))
    bq = np.ones((d_attn, 1))

    Wk = np.ones((d_attn, d_z))
    bk = np.ones((d_attn, 1))

    Wv = np.ones((d_out, d_z))
    bv = np.ones((d_out, 1))

    w = (Wq, bq, Wk, bk, Wv, bv)

    c = attention(X, Z, w)

    return c

In [54]:
test_attention()

array([[1.39870709, 1.49586079, 1.49622939],
       [1.39978232, 1.38302852, 1.76783427]])

In [55]:
test_ones()

array([[3., 3., 3.],
       [3., 3., 3.]])

In [56]:
test_zeros()

array([[0., 0., 0.],
       [0., 0., 0.]])

#ALGORITHM 5 : MHATTENTION

In [57]:
import numpy as np

def softmax(x, axis=0):
    x = x - np.max(x, axis=axis, keepdims=True)
    exp_x = np.exp(x)
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)


def attention(X, Z, params, mask=None):
    Wq, bq = params["Wq"], params["bq"]
    Wk, bk = params["Wk"], params["bk"]
    Wv, bv = params["Wv"], params["bv"]

    d_attn = Wq.shape[0]

    #Q, K and V
    Q = Wq @ X + bq[:, None]
    K = Wk @ Z + bk[:, None]
    V = Wv @ Z + bv[:, None]

    #Calculating attention scores
    scores = (Q.T @ K) / np.sqrt(d_attn)

    # Mask shape: (n_z, n_x) 1 = allowed, 0 = blocked

    if mask is not None:
        if mask.shape != (Z.shape[1], X.shape[1]):
            raise ValueError("Invalid mask dimensions")

        allowed = mask.T.astype(bool)

        if np.any(~np.any(allowed, axis=1)):
            raise ValueError("Every query must have at least one allowed key")

        scores = np.where(allowed, scores, -1e9)

    # attention weights
    weights = softmax(scores, axis=1)

    Y = V @ weights.T

    return Y


def mh_attention(X, Z, W, mask=None):
    heads = W["heads"]
    H = len(heads)
    Y_heads = []

    for h in range(H):
        Y_h = attention(X, Z, heads[h], mask)
        Y_heads.append(Y_h)

    Y = np.concatenate(Y_heads, axis=0)

    Wo = W["Wo"]
    bo = W["bo"]

    V_tilde = Wo @ Y + bo[:, None]
    return V_tilde

np.random.seed(42)

dx = 4
dz = 4
nx = 3
nz = 3
d_attn = 2
d_mid = 2
d_out = 4
H = 2

X = np.random.randn(dx, nx)
Z = np.random.randn(dz, nz)

heads = []

for h in range(H):
    heads.append({
        "Wq": np.random.randn(d_attn, dx),
        "bq": np.zeros(d_attn),
        "Wk": np.random.randn(d_attn, dz),
        "bk": np.zeros(d_attn),
        "Wv": np.random.randn(d_mid, dz),
        "bv": np.zeros(d_mid)})

W = {"heads": heads, "Wo": np.random.randn(d_out, H * d_mid), "bo": np.zeros(d_out)}

mask = np.ones((nz, nx))
result = mh_attention(X, Z, W, mask)
print("Multi-Head Attention Output:")
print(result)
print("\nOutput Shape:", result.shape)

Multi-Head Attention Output:
[[ 5.86662023  6.00484589  4.56644943]
 [ 0.21775779  1.39043774  2.87788684]
 [-1.29034789 -2.80227708 -2.01259033]
 [-1.73555506 -1.89253536 -1.81750474]]

Output Shape: (4, 3)


In [58]:
import unittest
import numpy as np


class TestMultiHeadAttention(unittest.TestCase):

    def setUp(self):
        np.random.seed(42)

        self.dx = 4
        self.dz = 4
        self.nx = 3
        self.nz = 3
        self.d_attn = 2
        self.d_mid = 2
        self.d_out = 4
        self.H = 2

        self.X = np.random.randn(self.dx, self.nx)
        self.Z = np.random.randn(self.dz, self.nz)

        self.heads = []

        for h in range(self.H):
            self.heads.append({
                "Wq": np.random.randn(self.d_attn, self.dx),
                "bq": np.zeros(self.d_attn),
                "Wk": np.random.randn(self.d_attn, self.dz),
                "bk": np.zeros(self.d_attn),
                "Wv": np.random.randn(self.d_mid, self.dz),
                "bv": np.zeros(self.d_mid)
            })

        self.W = {
            "heads": self.heads,
            "Wo": np.random.randn(
                self.d_out, self.H * self.d_mid
            ),
            "bo": np.zeros(self.d_out)
        }

        self.mask = np.ones((self.nz, self.nx))

    # Test 1: Single-head attention
    def test_single_head(self):
        result = attention(
            self.X, self.Z, self.heads[0], self.mask
        )

        self.assertEqual(result.shape, (self.d_mid, self.nx))

    # Test 2: Multi-head attention
    def test_multi_head(self):
        result = mh_attention(
            self.X, self.Z, self.W, self.mask
        )

        self.assertEqual(result.shape, (self.d_out, self.nx))

    # Test 3: Output must contain finite values
    def test_output_values(self):
        result = mh_attention(
            self.X, self.Z, self.W, self.mask
        )

        self.assertTrue(np.all(np.isfinite(result)))

    # Test 4: Masked tokens should not affect output
    def test_masked_attention(self):
        mask = np.ones((self.nz, self.nx))
        mask[2, :] = 0

        result1 = attention(
            self.X, self.Z, self.heads[0], mask
        )

        modified_Z = self.Z.copy()
        modified_Z[:, 2] += 100

        result2 = attention(
            self.X, modified_Z, self.heads[0], mask
        )

        np.testing.assert_allclose(result1, result2)

    # Test 5: Invalid mask dimensions
    def test_invalid_mask(self):
        wrong_mask = np.ones((2, 2))

        with self.assertRaises(ValueError):
            attention(
                self.X, self.Z, self.heads[0], wrong_mask
            )

    # Test 6: Fully masked query
    def test_fully_masked_query(self):
        mask = np.ones((self.nz, self.nx))
        mask[:, 0] = 0

        with self.assertRaises(ValueError):
            attention(
                self.X, self.Z, self.heads[0], mask
            )


if __name__ == "__main__":
  unittest.main(argv=['first-arg-is-ignored'], exit=False)

..................
----------------------------------------------------------------------
Ran 18 tests in 0.022s

OK
